<a href="https://colab.research.google.com/github/fariahasan00/Bangla-SLM-498R/blob/main/BanglaSLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Creating directory

In [ ]:
import os

project_dir = "/content/drive/MyDrive/Bangla_SLM_15M"
checkpoint_dir = os.path.join(project_dir, "checkpoints")
os.makedirs(checkpoint_dir, exist_ok=True)
print(f"Project directory set up at: {project_dir}")

Project directory set up at: /content/drive/MyDrive/Bangla_SLM_15M


In [ ]:
!pip install -q huggingface_hub pyarrow

In [ ]:
import os
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download, list_repo_files

REPO_ID = "hishab/titulm-bangla-corpus"
LOCAL_DIR = "/content/local_cache"
DRIVE_DIR = "/content/drive/MyDrive/Bangla_SLM_15M/data"

TARGET_TOKENS = 600_000_000       # derived from ~12.39M non-embedding params × ~48 tok/param
CHARS_PER_TOKEN_EST = 4.0         # PLACEHOLDER — recalibrate once your 6k-vocab tokenizer exists (see note below)
BATCH_SIZE = 10_000

os.makedirs(LOCAL_DIR, exist_ok=True)
os.makedirs(DRIVE_DIR, exist_ok=True)
local_out = os.path.join(LOCAL_DIR, "bangla_corpus_common_crawl.txt")
drive_out = os.path.join(DRIVE_DIR, "bangla_corpus_common_crawl.txt")
progress_log = os.path.join(LOCAL_DIR, "progress.log")

all_files = list_repo_files(REPO_ID, repo_type="dataset")
shards = sorted(f for f in all_files if f.startswith("common_crawl/") and f.endswith(".parquet"))
if not shards:
    raise RuntimeError("No common_crawl parquet files found — repo layout may have changed")
print(f"{len(shards)} shards available in common_crawl/")

done_shards = set()
total_chars = 0
if os.path.exists(progress_log):
    with open(progress_log) as f:
        for line in f:
            shard, chars = line.strip().split("\t")
            done_shards.add(shard)
            total_chars += int(chars)
    print(f"Resuming: {len(done_shards)} shards done, ~{int(total_chars/CHARS_PER_TOKEN_EST):,} tokens so far")

target_chars = TARGET_TOKENS * CHARS_PER_TOKEN_EST

with open(local_out, "a", encoding="utf-8") as out_f, open(progress_log, "a") as log_f:
    for i, shard_path in enumerate(shards):
        if total_chars >= target_chars:
            print(f"\nTarget reached: ~{int(total_chars/CHARS_PER_TOKEN_EST):,} tokens. {len(shards)-i} shards left untouched.")
            break
        if shard_path in done_shards:
            continue

        print(f"[{i+1}/{len(shards)}] {shard_path} ({int(total_chars/CHARS_PER_TOKEN_EST):,}/{TARGET_TOKENS:,} tokens so far)")
        local_parquet = hf_hub_download(repo_id=REPO_ID, filename=shard_path, repo_type="dataset")

        pf = pq.ParquetFile(local_parquet)
        if "text" not in pf.schema_arrow.names:
            raise ValueError(f"Unexpected schema in {shard_path}: {pf.schema_arrow.names}")

        shard_chars = 0
        for batch in pf.iter_batches(batch_size=BATCH_SIZE, columns=["text"]):
            texts = batch.column("text").to_pylist()
            lines = []
            for t in texts:
                if not t:
                    continue
                t = t.replace("\n", " ").strip()
                if t:
                    lines.append(t)
                    shard_chars += len(t)
            if lines:
                out_f.write("\n".join(lines) + "\n")

        total_chars += shard_chars
        log_f.write(f"{shard_path}\t{shard_chars}\n")
        log_f.flush()
        os.remove(local_parquet)

print(f"\nDone. Total chars: {total_chars:,} | est. tokens: {int(total_chars/CHARS_PER_TOKEN_EST):,}")

import shutil
shutil.copy(local_out, drive_out)
print(f"Copied to {drive_out}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


590 shards available in common_crawl/
[1/590] common_crawl/filtered-00000-of-00295.parquet (0/600,000,000 tokens so far)


common_crawl/filtered-00000-of-00295.par(…): reconstructing file:   0%|          |  0.00B /  177MB            

common_crawl/filtered-00000-of-00295.par(…): downloading bytes:           |  0.00B            

[2/590] common_crawl/filtered-00001-of-00295.parquet (46,948,568/600,000,000 tokens so far)


...

common_crawl/filtered-00011-of-00295.par(…): downloading bytes:           |  0.00B            

[13/590] common_crawl/filtered-00012-of-00295.parquet (559,996,246/600,000,000 tokens so far)


common_crawl/filtered-00012-of-00295.par(…): reconstructing file:   0%|          |  0.00B /  159MB            

common_crawl/filtered-00012-of-00295.par(…): downloading bytes:           |  0.00B            


Target reached: ~601,836,793 tokens. 577 shards left untouched.

Done. Total chars: 2,407,347,174 | est. tokens: 601,836,793
Copied to /content/drive/MyDrive/Bangla_SLM_15M/data/bangla_corpus_common_crawl.txt


In [ ]:
import os
path = "/content/drive/MyDrive/Bangla_SLM_15M/data/bangla_corpus_common_crawl.txt"
print(os.path.exists(path))
print(os.path.getsize(path) / 1e9, "GB")

True
6.371193872 GB


In [2]:
!pip install sentencepiece

In [3]:
import os
import random
import zlib
import numpy as np
import sentencepiece as spm
from tqdm import tqdm

#Configuration
DATA_DIR = "/content/drive/MyDrive/Bangla_SLM_15M/data"
INPUT_TXT = os.path.join(DATA_DIR, "bangla_corpus_common_crawl.txt")
SAMPLE_TXT = os.path.join(DATA_DIR, "tokenizer_sample.txt")
TOKENIZER_DIR = os.path.join(DATA_DIR, "tokenizer")
MODEL_PREFIX = os.path.join(TOKENIZER_DIR, "bangla_sp")
VOCAB_SIZE = 8000
VAL_RATIO = 0.01
BATCH_SIZE = 50_000
VAL_HASH_THRESHOLD = int(VAL_RATIO * (2**32))

os.makedirs(TOKENIZER_DIR, exist_ok=True)
train_bin_path = os.path.join(DATA_DIR, "train.bin")
val_bin_path = os.path.join(DATA_DIR, "val.bin")

#Training tokenizer on RAM safe Sample
model_path = MODEL_PREFIX + ".model"
if not os.path.exists(model_path):
    # Prepare a ~200,000 line sample (~100-200MB) to prevent Colab RAM crashes
    if not os.path.exists(SAMPLE_TXT):
        print("Preparing a text sample for SentencePiece training...")
        sample_size = 200_000
        with open(INPUT_TXT, "r", encoding="utf-8") as f_in:
            # Reservior sampling / top lines
            lines = []
            for idx, line in enumerate(f_in):
                if idx < sample_size:
                    lines.append(line)
                else:
                    break
        with open(SAMPLE_TXT, "w", encoding="utf-8") as f_out:
            f_out.writelines(lines)

    print("Training SentencePiece BPE tokenizer...")
    spm.SentencePieceTrainer.train(
        input=SAMPLE_TXT,
        model_prefix=MODEL_PREFIX,
        vocab_size=VOCAB_SIZE,
        model_type="bpe",
        character_coverage=1.0,   # full coverage for Bangla script
        pad_id=3, unk_id=0, bos_id=1, eos_id=2,
        input_sentence_size=200000,
        shuffle_input_sentence=True
    )
    print(f"Tokenizer saved to {model_path}")
    if os.path.exists(SAMPLE_TXT):
        os.remove(SAMPLE_TXT)  # Clean up temporary sample
else:
    print(f"Loading existing tokenizer from {model_path}...")

sp = spm.SentencePieceProcessor(model_file=model_path)
eos_id = sp.eos_id()

#Tokenize full dataset
train_f = open(train_bin_path, "wb")
val_f = open(val_bin_path, "wb")
train_token_count = 0
val_token_count = 0

def flush_batch(lines):
    global train_token_count, val_token_count
    if not lines:
        return
    encodings = sp.encode(lines, out_type=int)
    train_buf, val_buf = [], []
    for line, ids in zip(lines, encodings):
        ids = ids + [eos_id]
        h = zlib.crc32(line.encode("utf-8"))
        if h < VAL_HASH_THRESHOLD:
            val_buf.extend(ids)
        else:
            train_buf.extend(ids)
    if train_buf:
        arr = np.array(train_buf, dtype=np.uint16)
        arr.tofile(train_f)
        train_token_count += len(arr)
    if val_buf:
        arr = np.array(val_buf, dtype=np.uint16)
        arr.tofile(val_f)
        val_token_count += len(arr)

lines_batch = []
with open(INPUT_TXT, "r", encoding="utf-8") as f:
    for line in tqdm(f, desc="Tokenizing"):
        line_str = line.strip()
        if not line_str:
            continue
        lines_batch.append(line_str)
        if len(lines_batch) >= BATCH_SIZE:
            flush_batch(lines_batch)
            lines_batch = []
    flush_batch(lines_batch)

train_f.close()
val_f.close()

print(f"\n✓ train.bin: {train_token_count:,} tokens ({os.path.getsize(train_bin_path)/1e6:.2f} MB)")
print(f"✓ val.bin:   {val_token_count:,} tokens ({os.path.getsize(val_bin_path)/1e6:.2f} MB)")
print(f"  actual val ratio: {val_token_count/(train_token_count+val_token_count):.4f}")

Preparing a text sample for SentencePiece training...
Training SentencePiece BPE tokenizer...
Tokenizer saved to /content/drive/MyDrive/Bangla_SLM_15M/data/tokenizer/bangla_sp.model


Tokenizing: 1071330it [20:03, 890.24it/s] 



✓ train.bin: 629,166,706 tokens (1258.33 MB)
✓ val.bin:   6,380,700 tokens (12.76 MB)
  actual val ratio: 0.0100


vieweing vocabulary

In [4]:
import os
import sentencepiece as spm
import pandas as pd

MODEL_PATH = "/content/drive/MyDrive/Bangla_SLM_15M/data/tokenizer/bangla_sp.model"
sp = spm.SentencePieceProcessor(model_file=MODEL_PATH)

# Fetch all tokens and IDs
vocab = []
for id in range(sp.get_piece_size()):
    vocab.append({
        "Token ID": id,
        "Token Piece": sp.id_to_piece(id),
        "Is Unknown": sp.is_unknown(id),
        "Is Control": sp.is_control(id)
    })

# Convert to DataFrame for easy viewing/filtering
df_vocab = pd.DataFrame(vocab)

# Print special tokens and first 30 subwords
print("Total Vocab Size:", len(df_vocab))
print(df_vocab.head(30))

Total Vocab Size: 8000
    Token ID Token Piece  Is Unknown  Is Control
0          0       <unk>        True       False
1          1         <s>       False        True
2          2        </s>       False        True
3          3       <pad>       False        True
4          4          য়       False       False
5          5          ার       False       False
6          6          ▁স       False       False
7          7          ▁ক       False       False
8          8          ▁ব       False       False
9          9          ▁প       False       False
10        10          ের       False       False
11        11          ্র       False       False
12        12          ান       False       False
13        13          ্য       False       False
14        14          ▁ম       False       False
15        15          ▁এ       False       False
16        16          ▁আ       False       False
17        17          ▁হ       False       False
18        18          ▁ন       False       Fal

In [5]:
sample_sentence = "বাংলাদেশের অর্থনীতি প্রতিনিয়ত অগ্রসর হচ্ছে।"

# Encode to IDs and Token Strings
token_ids = sp.encode(sample_sentence, out_type=int)
token_pieces = sp.encode(sample_sentence, out_type=str)

# Display side-by-side
print(f"{'Token Piece':<20} | {'Token ID':<10}")
print("-" * 35)
for piece, tid in zip(token_pieces, token_ids):
    print(f"{piece:<20} | {tid:<10}")

Token Piece          | Token ID  
-----------------------------------
▁বাংলাদেশের          | 1034      
▁অর্থনীতি            | 6135      
▁প্রতিনি             | 815       
য়ত                  | 6554      
▁অগ্র                | 3235      
সর                   | 2470      
▁হচ্ছে               | 481       
।                    | 6946      
